In [ ]:
# >>> FIX (units / scale-invariance) — Yule-Walker Phastimate kept faithful ==================
#  Root cause of the alpha 'collapse': this cache stores EEG in VOLTS (~1e-6), so the per-window
#  variance r[0] ~ 1e-12 tripped aryule's ABSOLUTE guard `if e<=1e-12: return a`, which returned a
#  zero AR model on (essentially) every trial. A zero model -> zero forward prediction -> the phase
#  was read off a flatlined tail: PLV ~0.18, a frozen ~+104 deg bias, ~44% acc, identical across ALL
#  ar_order/window/edge/offset. It was never the algorithm, the estimator, or resting alpha.
#  This fix changes ONLY scale handling — the predictor, GA, fitness (1-PLV), bounds, and the
#  Levinson-Durbin recursion are untouched, so AR coefficients are identical on correctly-scaled data:
#    1. aryule: degeneracy guard is now scale-free (r[0]<=0) and the ACF is normalized so the 1e-12
#       tolerance is units-invariant (coeffs unchanged; this is what MATLAB aryule effectively does).
#    2. loop: epochs auto-scaled Volts->microvolts at load (TESA/Kirchhoff convention), which also
#       keeps phastimate's amplitude guards correctly scaled.
#
#  SPEED (fidelity-safe; revert for the byte-faithful reference):
#    GA_POP 100->50 and OPT_MAX_TRIALS=60 cut optimizer cost to ~2-3 min/subject (was ~20).
#    A 4-gene integer search does not need pop 100, and 60 high-amplitude cal trials give a
#    stable PLV. For the exact reference: set GA_POP=100 and OPT_MAX_TRIALS=10**9 (no cap).
# ============================================================================================
# =============================================================
# PHASTIMATE (Zrenner et al. 2020, github.com/bnplab/phastimate) on LEMON Mu/Alpha
#   faithful phastimate.m + the phastimate_optimize.m GENETIC ALGORITHM, ported from the
#   theta/onset notebook to the LEMON mu/alpha setup so it sits alongside EPN / PEAP /
#   PHASTPADDING / BOB: same LEMON cache, same FIR-231 9-13 Hz GT, same eval times, same
#   metrics, same JSON layout (join on 'name'). The predictor + GA are UNCHANGED.
# =============================================================
# Predictor (verbatim phastimate.m): demean window -> filtfilt(windowed-FIR, Hamming) ->
#   drop `edge` samples each side -> aryule(ord) -> forward-predict edge+ceil(hwin/2) ->
#   Hilbert last hwin samples -> phase at "now". Optimization: integer GENETIC ALGORITHM
#   (reimplements MATLAB ga from phastimate_optimize.m) over x=[window,filter_order,edge,ar_order],
#   constraint window>3*filter_order, fitness = circular variance (1-PLV), on the upper-amplitude
#   half of the cal epochs (includemask). Reports a fixed 'default' and the GA 'opt' (BOB schema).
#
# The LEMON port changes ONLY the framework, never the algorithm:
#   • data       : LEMON .npz cache (single-channel C3 Hjorth, 2D epochs, native 1000 Hz,
#                  resting -> PRE_SAMP=0); chronological cal/test split (first cal_n = cal).
#   • band / GT  : 4-8 Hz FIR-175  ->  9-13 Hz FIR-231 (byte-identical to EPN/PEAP/PHASTPADDING GT).
#   • eval times : [0.3..0.7] (onset-locked)  ->  [1.5..3.5] (from epoch start).
#   • SNR        : precomputed ch_theta_snr  ->  alpha_snr_per_epoch (9-13/1-40 power), like EPN/PEAP.
#   • params     : RESTORED to the 1 kHz/alpha originals the theta nb was scaled down from
#                  (GA bounds [400-750,100-250,30-120,5-60]; HILBERTWIN 128). The headline 'opt'
#                  result is whatever the GA finds inside these bounds, independent of 'default'.
#   • band mode  : fixed 9-13 (== GT), paralleling PHASTPADDING. Zrenner's alpha main_script
#                  individualizes to peak+/-1; toggle PHAST_INDIVIDUALIZE=True to restore that.
#   • output     : result_alpha/PHASTIMATE_lemon/phastimate_alpha_results.json,
#                  + n_train_samples (GA includemask budget) and compute_time_ms, to match the family.
#   theta_* descriptors are renamed alpha_*. Headline = opt_* (BOB-parallel); default = fixed reference.

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import firwin, filtfilt, hilbert, welch
from scipy.stats import wilcoxon, ttest_rel, pearsonr, spearmanr
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(Skipping Drive mount: {e})")

# ---- DATA + OUTPUT (LEMON, matched to EPN/PEAP/PHASTPADDING) ----
LOAD_DIR = '/content/drive/MyDrive/RAIN_IRC/LEMON'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/result_alpha/PHASTIMATE_lemon'   # parallel to EPN_lemon / PEAP_lemon
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(LOAD_DIR, '*.npz')))
assert files, f"no npz found in {LOAD_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = 0; _d0.close()      # resting epochs: t=0 at epoch start

EVAL_TIMES_S    = [1.5, 2.0, 2.5, 3.0, 3.5]
EVAL_TIMES_SAMP = [PRE_SAMP + int(round(t * FS)) for t in EVAL_TIMES_S]
ALPHA_SEARCH_LO, ALPHA_SEARCH_HI = 7.0, 14.0

# --- GT: 9-13 Hz zero-phase FIR-231 (filtfilt), identical taps to EPN/PEAP/PHASTPADDING ---
GT_BAND, GT_FIR_NUMTAPS = [9.0, 13.0], 231
gt_fir_b = firwin(GT_FIR_NUMTAPS, GT_BAND, fs=FS, window='hamming', pass_zero=False)

# --- phastimate constants (1 kHz / alpha; restored from the theta-scaled values) ---
HILBERTWIN = 128                     # phastimate alpha Hilbert window (power of 2); theta nb used 64 @300 Hz
# Internal FIR passband. Default fixed 9-13 (== GT), paralleling PHASTPADDING. Set
# PHAST_INDIVIDUALIZE=True for Zrenner's alpha main_script behaviour (band = peak +/- PHAST_PEAK_HALFWIDTH).
PHAST_BAND           = [9.0, 13.0]
PHAST_INDIVIDUALIZE  = False
PHAST_PEAK_HALFWIDTH = 1.0           # Hz, used only when PHAST_INDIVIDUALIZE
DEFAULT = {'window': 600, 'filter_order': 150, 'edge': 64, 'ar_order': 30}   # fixed reference (in-bounds, 1 kHz)

# --- genetic algorithm (phastimate_optimize.m: ga, PopulationSize 100, intcon, constraint) ---
# genes x = [window_length, filter_order, edge, ar_order]   (phastimate's order)
GA_BOUNDS  = [(400, 750), (100, 250), (30, 120), (5, 60)]   # 1 kHz/alpha originals (theta nb scaled these down)
GA_POP, GA_MAXGEN, GA_STALL = 50, 12, 6                    # pop matches phastimate; MAXGEN<<ga-default(400) for the sweep
GA_ELITE, GA_XFRAC, GA_MUT, GA_SEED = 5, 0.8, 0.12, 42      # elite=ceil(.05*100); crossover .8 (ga defaults)
OPT_AMP_FRACTION = 0.5                                      # includemask: optimize on upper-amplitude-half epochs
OPT_MAX_TRIALS   = 60                                       # [SPEED] cap GA fitness to <=this many high-amp cal trials

print(f"\n{'='*88}")
print(f"PHASTIMATE (Zrenner 2020) on LEMON Mu/Alpha — {FS:.0f} Hz, FIR-231 GT")
print(f"{'='*88}")
_bandtxt = (f"peak+/-{PHAST_PEAK_HALFWIDTH} (individualized)" if PHAST_INDIVIDUALIZE else f"FIXED {PHAST_BAND} (== GT)")
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | HILBERTWIN={HILBERTWIN} | internal band = {_bandtxt}")
print(f"  Genetic algorithm: genes [window,filter_order,edge,ar_order], pop={GA_POP}, maxgen={GA_MAXGEN}")
print(f"    constraint window>3*filter_order | fitness = circular variance (1-PLV), NOT MACE")
print(f"    optimized on upper-{int((1-OPT_AMP_FRACTION)*100)}%-amplitude cal epochs (includemask)")
print(f"  bounds {GA_BOUNDS}")
print(f"  GT: firwin({GT_FIR_NUMTAPS}, {GT_BAND}, hamming, fs={FS:.0f}) -> filtfilt -> Hilbert")
print(f"  Data: {LOAD_DIR}  ({len(files)} subjects) | Output: {OUTDIR}")


# =============================================================
# phastimate internals  (faithful to phastimate.m)              [ALGORITHM UNCHANGED]
# =============================================================

def aryule(x, order):
    """MATLAB aryule: Levinson-Durbin on the biased autocorrelation.
    Returns a=[1,a1,...,ap] with x[n] = -sum_k a[k] x[n-k]."""
    x = np.asarray(x, np.float64); n = len(x)
    r = np.correlate(x, x, 'full')[n - 1:n + order] / n
    a = np.zeros(order + 1); a[0] = 1.0
    if r[0] <= 0.0:                      # [FIX] scale-free degeneracy guard (was absolute 'e<=1e-12'
        return a                         #       variance floor -> nuked Volt-scale EEG into AR-of-zeros)
    r = r / r[0]; e = 1.0                # [FIX] normalize ACF -> 1e-12 tolerance is units-invariant; AR coeffs unchanged
    for m in range(1, order + 1):
        acc = r[m] + (np.dot(a[1:m], r[1:m][::-1]) if m > 1 else 0.0)
        k = -acc / e
        ao = a[1:m].copy(); a[1:m] = ao + k * ao[::-1]; a[m] = k
        e *= (1.0 - k * k)
        if e <= 1e-12:
            break
    return a

def design_phastimate_filter(filter_order, peak, fs):
    """designfilt('bandpassfir','FilterOrder',ord,'window'); taps=ord+1.
    Default band = fixed PHAST_BAND (== GT). If PHAST_INDIVIDUALIZE, band = peak +/- PHAST_PEAK_HALFWIDTH
    (Zrenner alpha main_script)."""
    if PHAST_INDIVIDUALIZE and np.isfinite(peak):
        lo, hi = float(peak) - PHAST_PEAK_HALFWIDTH, float(peak) + PHAST_PEAK_HALFWIDTH
    else:
        lo, hi = PHAST_BAND
    if hi <= lo + 0.2 or int(filter_order) + 1 < 4:
        return None
    return firwin(int(filter_order) + 1, [lo, hi], fs=fs, window='hamming', pass_zero=False)

def phastimate(window_data, fir_b, edge, ord, hilbertwindow, offset=0, iterations=None):
    """Verbatim port of phastimate.m. window_data: 1D, last sample = 'now'."""
    if iterations is None:
        iterations = edge + int(np.ceil(hilbertwindow / 2.0))
    x = np.asarray(window_data, np.float64); x = x - np.mean(x)          # detrend(...,'constant')
    try:
        padlen = min(3 * len(fir_b), len(x) - 1)
        xf = filtfilt(fir_b, [1.0], x, padlen=padlen)                    # filtfilt(D,data)
    except Exception:
        return np.nan
    xfe = xf[edge:len(xf) - edge]                                        # remove edge both ends
    if len(xfe) < ord + 5 or np.std(xfe) < 1e-10:
        return np.nan
    a = aryule(xfe, ord); coeffs = -np.flip(a[1:])                       # -1*flip(a(2:end))
    L = len(xfe); ext = np.empty(L + iterations); ext[:L] = xfe
    for k in range(iterations):                                         # forward prediction
        p = L + k; ext[p] = np.dot(coeffs, ext[p - ord:p])
    if hilbertwindow > len(ext):
        return np.nan
    seg = ext[len(ext) - hilbertwindow:]                                # last hilbertwindow samples
    analytic = hilbert(seg)
    idx = hilbertwindow - iterations + edge - 1 + offset                # MATLAB end-iterations+edge -> "now"
    if idx < 0 or idx >= len(analytic):
        return np.nan
    return float(np.angle(analytic[idx]))

def phastimate_at(raw_1d, eval_samp, window, fir_b, edge, ord, hwin):
    start = eval_samp - window + 1
    if start < 0:
        return np.nan
    return phastimate(raw_1d[start:eval_samp + 1], fir_b, int(edge), int(ord), int(hwin))


# =============================================================
# Genetic algorithm  (reimplements MATLAB `ga` from phastimate_optimize.m)   [UNCHANGED]
# =============================================================

def genetic_optimize(fitness, bounds, pop_size=100, max_gen=12, elite=5,
                     crossover_frac=0.8, mut_prob=0.12, stall_gen=6, seed=42, x0=None):
    rng = np.random.default_rng(seed); nv = len(bounds)
    lo = np.array([b[0] for b in bounds], float); hi = np.array([b[1] for b in bounds], float)
    loi, hii = lo.astype(int), hi.astype(int)
    pop = [np.round(lo + rng.random(nv) * (hi - lo)).astype(int) for _ in range(pop_size)]
    if x0 is not None:
        pop[0] = np.clip(np.round(np.array(x0)).astype(int), loi, hii)
    fit = np.array([fitness(ind) for ind in pop]); hist = []; gens = 0
    for _ in range(max_gen):
        gens += 1
        idx = np.argsort(fit); pop = [pop[i] for i in idx]; fit = fit[idx]
        hist.append(float(fit[0]))
        if len(hist) > stall_gen and (hist[-stall_gen - 1] - hist[-1] < 1e-6):
            break
        newpop = [pop[i].copy() for i in range(min(elite, pop_size))]     # elitism
        while len(newpop) < pop_size:
            a, b = rng.integers(0, pop_size, 2); p1 = pop[a] if fit[a] <= fit[b] else pop[b]
            a, b = rng.integers(0, pop_size, 2); p2 = pop[a] if fit[a] <= fit[b] else pop[b]
            if rng.random() < crossover_frac:                            # scattered crossover
                mask = rng.random(nv) < 0.5; child = np.where(mask, p1, p2).astype(float)
            else:
                child = p1.astype(float).copy()
            mut = rng.random(nv) < mut_prob                              # mutation
            if mut.any():
                child[mut] = lo[mut] + rng.random(int(mut.sum())) * (hi[mut] - lo[mut])
            newpop.append(np.clip(np.round(child).astype(int), loi, hii))
        pop = newpop; fit = np.array([fitness(ind) for ind in pop])
    idx = np.argsort(fit)
    return pop[idx[0]], float(fit[idx[0]]), gens


# =============================================================
# GT (+amplitude for includemask) + metrics + alpha descriptors  (EPN/PEAP/BOB-IDENTICAL)
# =============================================================

def gt_analytic_fir(epoch):
    try:
        padlen = min(3 * len(gt_fir_b), len(epoch) - 1)
        an = hilbert(filtfilt(gt_fir_b, [1.0], epoch.astype(np.float64), padlen=padlen))
        return np.angle(an), np.abs(an)
    except Exception:
        nan = np.full(len(epoch), np.nan); return nan, nan

def circ(a, b):
    return np.angle(np.exp(1j * (a - b)))

def errors_to_metrics(errors):
    errors = np.asarray(errors, float); errors = errors[np.isfinite(errors)]
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    ma = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(ma)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),
            'acc': float((1.0 - ma / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_at(ph_list, trials, esamp):
    vals = [ph_list[tr][esamp] for tr in trials
            if esamp < len(ph_list[tr]) and np.isfinite(ph_list[tr][esamp])]
    return float(np.abs(np.mean(np.exp(1j * np.array(vals))))) if len(vals) >= 2 else float('nan')

def alpha_snr_per_epoch(raw_1ch):
    """mean over epochs of (9-13 Hz power / 1-40 Hz power); identical to EPN/PEAP / the LEMON loader's mu-SNR."""
    snrs = []
    for tr in range(raw_1ch.shape[0]):
        ep = raw_1ch[tr].astype(np.float64)
        psd = np.abs(np.fft.rfft(ep)) ** 2
        freqs = np.fft.rfftfreq(len(ep), 1.0 / FS)
        mu = psd[(freqs >= 9) & (freqs <= 13)].mean()
        broad = psd[(freqs >= 1) & (freqs <= 40)].mean()
        snrs.append(mu / broad if broad > 0 else 0.0)
    return float(np.mean(snrs))

def estimate_alpha_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16:
        return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= ALPHA_SEARCH_LO) & (f <= ALPHA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

def run_phast_per_tp(raw, ph_fir, trials, window, fir_b, edge, ord, epoch_len):
    per_tp = {t: [] for t in EVAL_TIMES_S}; n_eval = n_fail = 0
    for tr in trials:
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            if es >= epoch_len or es - window + 1 < 0:
                continue
            n_eval += 1
            est = phastimate_at(raw[tr], es, window, fir_b, edge, ord, HILBERTWIN)
            gf = ph_fir[tr][es]
            if not (np.isfinite(est) and np.isfinite(gf)):
                n_fail += 1; continue
            per_tp[t].append(circ(est, gf))
    return {t: np.array(v) for t, v in per_tp.items()}, (100.0 * n_fail / max(n_eval, 1))


Mounted at /content/drive

PHASTIMATE (Zrenner 2020) on LEMON Mu/Alpha — 1000 Hz, FIR-231 GT
  FS=1000 Hz | t=0 at sample 0 | HILBERTWIN=128 | internal band = FIXED [9.0, 13.0] (== GT)
  Genetic algorithm: genes [window,filter_order,edge,ar_order], pop=50, maxgen=12
    constraint window>3*filter_order | fitness = circular variance (1-PLV), NOT MACE
    optimized on upper-50%-amplitude cal epochs (includemask)
  bounds [(400, 750), (100, 250), (30, 120), (5, 60)]
  GT: firwin(231, [9.0, 13.0], hamming, fs=1000) -> filtfilt -> Hilbert
  Data: /content/drive/MyDrive/RAIN_IRC/LEMON  (20 subjects) | Output: /content/drive/MyDrive/RAIN_IRC/result_alpha/PHASTIMATE_lemon


In [ ]:
# =============================================================
# Main loop  [phastimate UNCHANGED; GA per subject on cal includemask, scored on LEMON TEST]
# =============================================================
print(f"\n{'Subj':<14s}{'N':>5s}{'defMACE':>8s}{'optMACE':>8s}{'optPLV':>7s}{'optCSD':>7s}{'optAcc':>7s}{'pk':>5s}{'SNR':>6s}{'ITPC':>6s}{'gen':>4s}{'t':>5s}")
print("-" * 100)

results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS:
        print(f"{name}: fs mismatch, skipping"); d.close(); continue
    raw = d['epochs'].astype(np.float64)               # LEMON: single channel, 2D (n_trials x n_samp)
    if np.nanmedian(np.nanstd(raw, axis=1)) < 1e-3:    # [FIX] cache in Volts -> microvolts (TESA/Kirchhoff units);
        raw = raw * 1e6                                #       keeps aryule + phastimate amplitude guards correctly scaled
    n_trials, epoch_len = raw.shape
    ch_name = str(d['montage']) if 'montage' in d.files else 'C3-Hjorth'
    cal_n = int(d['cal_n'])
    cal_trials  = np.arange(cal_n)                      # chronological split (first cal_n = cal)
    test_trials = np.arange(cal_n, n_trials)
    alpha_snr = alpha_snr_per_epoch(raw)

    ph_fir, amp_fir = [], []
    for tr in range(n_trials):
        p, a = gt_analytic_fir(raw[tr]); ph_fir.append(p); amp_fir.append(a)
    itpc_tp = {t: (itpc_at(ph_fir, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_alpha_peak(raw, PRE_SAMP)
    peak = f_peak if np.isfinite(f_peak) else 11.0     # phastimate individualization fallback (alpha center)

    # --- includemask: upper-amplitude-half cal epochs (mean GT amp over eval points) ---
    es_in = [es for es in EVAL_TIMES_SAMP if es < epoch_len]
    cal_amp = np.array([np.nanmean([amp_fir[tr][es] for es in es_in]) for tr in cal_trials])
    thr = np.nanquantile(cal_amp, OPT_AMP_FRACTION)
    opt_trials = cal_trials[cal_amp >= thr]
    if len(opt_trials) < 5:
        opt_trials = cal_trials
    if len(opt_trials) > OPT_MAX_TRIALS:                       # [SPEED] subsample high-amp cal trials
        opt_trials = opt_trials[np.linspace(0, len(opt_trials) - 1, OPT_MAX_TRIALS).astype(int)]

    # --- default (fixed, in-bounds reference) on TEST ---
    fb_def = design_phastimate_filter(DEFAULT['filter_order'], peak, FS)
    dflt_tp, dflt_fail = run_phast_per_tp(raw, ph_fir, test_trials, DEFAULT['window'], fb_def,
                                          DEFAULT['edge'], DEFAULT['ar_order'], epoch_len)
    dflt_pool = (np.concatenate([dflt_tp[t] for t in EVAL_TIMES_S])
                 if any(len(dflt_tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    dflt_m = errors_to_metrics(dflt_pool); dflt_tp_m = {t: errors_to_metrics(dflt_tp[t]) for t in EVAL_TIMES_S}

    # --- GENETIC ALGORITHM on cal includemask: minimize circular variance (1-PLV) ---
    def fitness(x):
        w, fo, ed, ar = int(x[0]), int(x[1]), int(x[2]), int(x[3])
        if w <= 3 * fo:                       # phastimate Aineq: window > 3*filter_order
            return 1.0
        if w - 2 * ed < ar + 5:               # predictor feasibility
            return 1.0
        fb = design_phastimate_filter(fo, peak, FS)
        if fb is None:
            return 1.0
        errs = []
        for tr in opt_trials:
            for es in es_in:
                if es - w + 1 < 0:
                    continue
                est = phastimate_at(raw[tr], es, w, fb, ed, ar, HILBERTWIN)
                gf = ph_fir[tr][es]
                if np.isfinite(est) and np.isfinite(gf):
                    errs.append(est - gf)
        if len(errs) < 10:
            return 1.0
        return float(1.0 - np.abs(np.mean(np.exp(1j * np.array(errs)))))   # circular variance

    x0 = [int(np.ceil((b[0] + b[1]) / 2)) for b in GA_BOUNDS]
    best_x, fval, n_gen = genetic_optimize(fitness, GA_BOUNDS, pop_size=GA_POP, max_gen=GA_MAXGEN,
                                           elite=GA_ELITE, crossover_frac=GA_XFRAC, mut_prob=GA_MUT,
                                           stall_gen=GA_STALL, seed=GA_SEED, x0=x0)
    _bl, _bh = ((peak - PHAST_PEAK_HALFWIDTH, peak + PHAST_PEAK_HALFWIDTH)
                if (PHAST_INDIVIDUALIZE and np.isfinite(peak)) else (PHAST_BAND[0], PHAST_BAND[1]))
    bp = {'window': int(best_x[0]), 'filter_order': int(best_x[1]), 'edge': int(best_x[2]),
          'ar_order': int(best_x[3]), 'band_low': float(_bl), 'band_high': float(_bh),
          'cal_circ_var': float(fval), 'cal_plv': float(1.0 - fval),
          'n_generations': int(n_gen), 'n_opt_trials': int(len(opt_trials))}

    # --- optimized on TEST ---
    fb_opt = design_phastimate_filter(bp['filter_order'], peak, FS)
    opt_tp, opt_fail = run_phast_per_tp(raw, ph_fir, test_trials, bp['window'], fb_opt,
                                        bp['edge'], bp['ar_order'], epoch_len)
    opt_pool = (np.concatenate([opt_tp[t] for t in EVAL_TIMES_S])
                if any(len(opt_tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    opt_m = errors_to_metrics(opt_pool); opt_tp_m = {t: errors_to_metrics(opt_tp[t]) for t in EVAL_TIMES_S}

    # --- per-call latency (opt params; analogue of EPN/PEAP/PHASTPADDING compute_time_ms) ---
    tsamps = []
    valid_es = [es for es in EVAL_TIMES_SAMP if es < epoch_len and es - bp['window'] + 1 >= 0]
    if len(test_trials) and valid_es and fb_opt is not None:
        tr0, es0 = int(test_trials[0]), valid_es[0]
        for _ in range(50):
            ts = time.perf_counter()
            phastimate_at(raw[tr0], es0, bp['window'], fb_opt, bp['edge'], bp['ar_order'], HILBERTWIN)
            tsamps.append((time.perf_counter() - ts) * 1000)
    n_train_samples = int(len(opt_trials) * len(es_in))    # GA includemask budget (PHASTIMATE trains/optimizes)

    dt = time.time() - t0; pk = f"{f_peak:.1f}" if np.isfinite(f_peak) else "n/a"
    print(f"{name:<14s}{opt_m['n_samples']:>5d}{dflt_m['mace']:>7.1f}\u00b0{opt_m['mace']:>7.1f}\u00b0"
          f"{opt_m['plv']:>7.3f}{opt_m['csd']:>6.1f}\u00b0{opt_m['acc']:>6.1f}%{pk:>5s}{alpha_snr:>6.2f}"
          f"{itpc_mean:>6.3f}{n_gen:>4d}{dt:>4.0f}s")

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': opt_m['n_samples'], 'n_train_samples': n_train_samples,
        'fail_pct': opt_fail, 'compute_time_ms': float(np.mean(tsamps)) if tsamps else 0.0,
        'alpha_snr': alpha_snr,
        'alpha_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'alpha_itpc_mean': itpc_mean,
        'alpha_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'best_params': bp, 'default_params': DEFAULT,
        'default_metrics_pooled': dflt_m,
        'default_metrics_per_timepoint': {str(t): dflt_tp_m[t] for t in EVAL_TIMES_S},
        'default_errors_per_timepoint_rad': {str(t): [float(x) for x in dflt_tp[t]] for t in EVAL_TIMES_S},
        'opt_metrics_pooled': opt_m,
        'opt_metrics_per_timepoint': {str(t): opt_tp_m[t] for t in EVAL_TIMES_S},
        'opt_errors_per_timepoint_rad': {str(t): [float(x) for x in opt_tp[t]] for t in EVAL_TIMES_S}})
    d.close(); del raw, ph_fir, amp_fir; gc.collect()

print(f"\n{len(results)} subjects done")



Subj              N defMACE optMACE optPLV optCSD optAcc   pk   SNR  ITPC gen    t
----------------------------------------------------------------------------------------------------
sub-032311      405   35.2°   35.8°  0.708  47.6°  80.1% 11.0  3.60 0.093  12 304s
sub-032317      410   40.5°   38.4°  0.680  50.3°  78.6% 10.1  3.60 0.083  12 309s
sub-032324      400   31.9°   30.0°  0.777  40.7°  83.3% 10.6  3.90 0.087  12 308s
sub-032330      430   38.9°   39.7°  0.655  52.7°  77.9% 11.1  3.76 0.134  12 324s
sub-032332      400   37.4°   34.3°  0.739  44.6°  80.9% 10.4  3.49 0.077  12 321s
sub-032355      430   40.9°   37.3°  0.696  48.8°  79.3% 10.4  3.52 0.077  12 314s
sub-032361      405   31.3°   33.1°  0.733  45.1°  81.6% 10.5  4.32 0.135  12 301s
sub-032370      400   32.3°   32.2°  0.758  42.7°  82.1% 10.9  3.58 0.122  12 311s
sub-032383      420   38.6°   31.0°  0.765  41.9°  82.8% 10.3  4.87 0.055  12 295s
sub-032394      420   43.8°   41.8°  0.619  56.1°  76.8% 11.3  3.96 

In [ ]:
# =============================================================
# SUMMARY + SAVE  (model-tagged JSON; BOB-parallel default vs opt)
# =============================================================
valid = [r for r in results if r['n_test_samples'] >= 10]; n = len(valid)
dM = np.array([r['default_metrics_pooled']['mace'] for r in valid])
oM = np.array([r['opt_metrics_pooled']['mace'] for r in valid])
oP = np.array([r['opt_metrics_pooled']['plv'] for r in valid])
oC = np.array([r['opt_metrics_pooled']['csd'] for r in valid])
oA = np.array([r['opt_metrics_pooled']['acc'] for r in valid])
dA = np.array([r['default_metrics_pooled']['acc'] for r in valid])
SNR = np.array([r['alpha_snr'] for r in valid]); IT = np.array([r['alpha_itpc_mean'] for r in valid])

print('=' * 72); print('PHASTIMATE ALPHA (mu) — DEFAULT vs GA-OPTIMIZED (genetic algorithm)'); print('=' * 72)
print(f"  Default   MACE {dM.mean():.1f}\u00b0\u00b1{dM.std():.1f}   Acc {dA.mean():.1f}%")
print(f"  Optimized MACE {oM.mean():.1f}\u00b0\u00b1{oM.std():.1f}   PLV {oP.mean():.3f}   CSD {oC.mean():.1f}\u00b0   Acc {oA.mean():.1f}%")
diff = dM - oM
print(f"  GA gain: {diff.mean():+.1f}\u00b0 MACE  ({int(np.sum(diff > 0))}/{n} improved)")
print(f"  mean failure rate {np.mean([r['fail_pct'] for r in results]):.1f}%")
print(f"  (n={n} participants; GA fitness = circular variance / 1-PLV, not MACE)")
if n >= 3 and np.any(diff != 0):
    _, w_p = wilcoxon(dM, oM); _, t_p = ttest_rel(dM, oM)
    print(f"  Paired default-vs-opt: Wilcoxon p={w_p:.4g} | t-test p={t_p:.4g}")

print('\n--- Per-timepoint (default vs optimized) ---')
print(f"{'ms':>5}{'defMACE':>9}{'optMACE':>9}{'optPLV':>8}{'optCSD':>8}{'optAcc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    dm = np.array([r['default_metrics_per_timepoint'][str(t)]['mace'] for r in valid])
    g = lambda k: np.array([r['opt_metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['alpha_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{dm.mean():>8.1f}\u00b0{g('mace').mean():>8.1f}\u00b0{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}\u00b0{g('acc').mean():>7.1f}%{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n),
               'mean_mace_default': float(dM.mean()), 'mean_mace_opt': float(oM.mean()),
               'median_mace_opt': float(np.median(oM)), 'mean_csd_opt': float(oC.mean()),
               'mean_acc_opt': float(oA.mean()), 'mean_plv_opt': float(oP.mean()),
               'mean_fail_pct': float(np.mean([r['fail_pct'] for r in results]))}

print('\n--- Accuracy (optimized) correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, oA); rs1, ps1 = spearmanr(SNR, oA)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, oA); rs2, ps2 = spearmanr(IT, oA)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

print('\n--- Per-subject GA-optimal phastimate params ---')
print(f"{'Subject':<14}{'window':>7}{'filtOrd':>8}{'edge':>5}{'arOrd':>6}{'band':>12}{'calPLV':>7}{'gen':>4}")
for r in valid:
    p = r['best_params']
    band_str = f"{p['band_low']:.1f}-{p['band_high']:.1f}"
    print(f"{r['name']:<14}{p['window']:>7}{p['filter_order']:>8}{p['edge']:>5}{p['ar_order']:>6}"
          f"{band_str:>12}{p['cal_plv']:>7.3f}{p['n_generations']:>4}")

out = {'model': 'PHASTIMATE',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': GT_BAND, 'gt_fir_numtaps': GT_FIR_NUMTAPS, 'hilbertwindow': HILBERTWIN,
                  'phast_band': PHAST_BAND, 'individualize': PHAST_INDIVIDUALIZE,
                  'peak_halfwidth_hz': PHAST_PEAK_HALFWIDTH,
                  'band_note': ('fixed 9-13 Hz (== GT), paralleling PHASTPADDING; set individualize=True for '
                                "Zrenner's alpha peak+/-1 main_script behaviour"),
                  'ga_bounds': GA_BOUNDS,
                  'ga_genes': ['window_length', 'filter_order', 'edge', 'ar_order'],
                  'ga_population': GA_POP, 'ga_maxgen': GA_MAXGEN,
                  'ga_constraint': 'window > 3*filter_order (phastimate Aineq)',
                  'ga_fitness': 'circular variance (1-PLV), phastimate native -- NOT MACE',
                  'ga_includemask': f'upper-{int((1-OPT_AMP_FRACTION)*100)}%-amplitude cal epochs',
                  'optimizer': 'genetic algorithm (reimplements MATLAB ga from phastimate_optimize.m)',
                  'algorithm': ('Zrenner2020 phastimate.m: windowed-FIR -> edge trim -> aryule -> '
                                'forward predict edge+ceil(hwin/2) -> Hilbert -> phase at now'),
                  'data_dir': LOAD_DIR, 'dataset': 'LEMON top-20 mu-SNR (C3 Hjorth)', 'band': 'mu/alpha 9-13 Hz',
                  'split': 'chronological cal/test (first cal_n epochs = cal)',
                  'channel': 'C3 Hjorth (single channel)', 'conditions': ['default', 'opt'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'note': ('phastimate ported from the theta/onset notebook to LEMON mu/alpha; GA bounds + '
                           'HILBERTWIN restored to the 1 kHz/alpha originals. headline = opt (GA) block, '
                           'analogue of BOB opt_* and of EPN/PEAP/PHASTPADDING metrics_pooled; default = fixed '
                           'in-bounds reference. alpha_snr / alpha_itpc_* / alpha_peak_hz computed identically to '
                           'EPN/PEAP -> match per subject across models. n_train_samples = GA includemask budget; '
                           'fail_pct = % eval points with no finite estimate. Same data/GT/split as the family -> '
                           'join on name.')},
       'group_stats': group_stats, 'subjects': results}
outpath = os.path.join(OUTDIR, 'phastimate_alpha_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: default + opt metrics, per-timepoint errors (rad), SNR/ITPC/peak, GA params, fail_pct, compute_time_ms.")
print("Same data/GT/split as EPN/PEAP/PHASTPADDING/BOB LEMON -> join on 'name';"
      " PHASTIMATE.opt_metrics_pooled <-> EPN/PEAP/PHASTPADDING.metrics_pooled <-> BOB.opt_metrics_pooled.")


PHASTIMATE ALPHA (mu) — DEFAULT vs GA-OPTIMIZED (genetic algorithm)
  Default   MACE 36.6°±5.8   Acc 79.7%
  Optimized MACE 34.3°±5.7   PLV 0.726   CSD 45.7°   Acc 81.0%
  GA gain: +2.3° MACE  (17/20 improved)
  mean failure rate 0.0%
  (n=20 participants; GA fitness = circular variance / 1-PLV, not MACE)
  Paired default-vs-opt: Wilcoxon p=0.0001335 | t-test p=0.0003306

--- Per-timepoint (default vs optimized) ---
   ms  defMACE  optMACE  optPLV  optCSD  optAcc    ITPC      N
 1500    37.3°    34.6°   0.726   45.4°   80.8%   0.081     84
 2000    38.0°    35.2°   0.714   46.8°   80.5%   0.113     84
 2500    36.5°    34.3°   0.732   44.9°   81.0%   0.086     84
 3000    35.0°    33.3°   0.736   44.7°   81.5%   0.102     84
 3500    36.1°    34.0°   0.731   45.1°   81.1%   0.104     84

--- Accuracy (optimized) correlations across subjects ---
  Acc vs SNR : r=+0.626 (p=0.0032)  rho=+0.439 (p=0.0528)
  Acc vs ITPC: r=-0.164 (p=0.4909)  rho=-0.045 (p=0.8502)

--- Per-subject GA-optimal